# Proprietary Momentum Strategy — DJIA 30, Apr 2016 – Apr 2026

**MIT 15.C51 Spring 2026 — Project #1 (Proprietary Trading track, momentum half).**

This notebook develops, backtests, iteratively improves, and documents a proprietary cross-sectional momentum strategy on the Dow Jones 30, culminating in an Investment Committee Memorandum (Stage 7). Mean reversion is a separate deliverable and is explicitly out of scope.

---

### How to run

1. **Kernel → Restart & Run All.** Every cell is designed to execute top-to-bottom from a fresh kernel.
2. Dependencies are installed inline in the first code cell. Required libraries: `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `yfinance`, `pandas_datareader`, `hmmlearn`, `statsmodels`, `xgboost`.
3. Data is fetched live (no cached files committed). Prices/volume via `yfinance`, risk-free rate via FRED `DTB3`, Fama-French factors via `pandas_datareader.famafrench`.
4. All tunable parameters live in the `RUN_CONFIG` dict in Stage 0 — do **not** introduce magic numbers elsewhere. Strategy functions read from `RUN_CONFIG` (passed as `cfg`).
5. Intermediate results are checkpointed to `results.pkl` after Stage 3 and Stage 5 so re-running is cheap.
6. Stage structure follows PROMPT.md §3 — each stage ends with a git commit tagged `stage-N: <what>`.

### Non-negotiables (baked into the pipeline)

- **No look-ahead.** Weights are always shifted by one trading day before being multiplied by realised returns.
- **Point-in-time universe.** The DJIA constituent map is reconstructed from `BASELINE_APR2016` + `CHANGES`; no survivorship bias.
- **Transaction costs.** Applied on portfolio turnover (`weights.diff().abs().sum(axis=1)`) at 10 bps base case. Sensitivities at 0/5/15/20 bps in Stage 6.
- **Walk-forward validation.** Expanding-window folds (see `RUN_CONFIG['WALK_FORWARD_FOLDS']`). No hyperparameter tuning on test windows.
- **Benchmarks.** Every strategy is compared against EW-DJIA buy-and-hold, `^DJI` (price-weighted), SPY, and compounded DTB3.

In [ ]:
# One-shot dependency install. Safe to re-run; pip is idempotent.
# hmmlearn / statsmodels / xgboost / pandas_datareader are not in Colab's default image.
!pip install --quiet hmmlearn statsmodels xgboost pandas_datareader

---

## Stage 0 — Setup & Reproducibility

Single source of truth for every parameter (`RUN_CONFIG`), pinned RNG seeds, version pins for the audit trail, and a global matplotlib style for publication-quality figures.

In [ ]:
from __future__ import annotations

import logging
import random
import sys
import warnings
from dataclasses import dataclass
from typing import Any

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('momentum')

In [ ]:
# ── RUN_CONFIG ────────────────────────────────────────────────────────────
# Single source of truth. Every downstream cell reads from this dict; no
# magic numbers elsewhere in the notebook.

SEED = 42

RUN_CONFIG: dict[str, Any] = {
    # Universe / horizon
    'START_DATE':        '2016-04-18',
    'END_DATE':          '2026-04-18',
    'FFILL_LIMIT_DAYS':  10,

    # Portfolio construction
    'REBALANCE_FREQ':    'ME',           # pandas month-end alias
    'LONG_PCT':          0.20,           # long top quintile
    'SHORT_PCT':         0.20,           # short bottom quintile

    # Transaction costs (per unit turnover, one-way → charged on abs Δweight)
    'COST_BPS':          10,             # base case
    'COST_BPS_GRID':     [0, 5, 10, 15, 20],  # Stage 6 sensitivity

    # Signal lookbacks (trading days)
    'LOOKBACKS': {
        'mom_long':      252,            # 12-month total window
        'mom_mid':       126,            # 6-month
        'mom_short':     63,             # 3-month
        'skip':          21,             # skip-one-month lag
        'vol_long':      60,             # vol estimator
        'vol_short':     20,             # short-horizon vol for scaling
        'ma_fast':       50,
        'ma_slow':       200,
        'beta_window':   252,            # residual-momentum regression window
        'vol_target':    0.10,           # annualised target for Stage 5 vol-targeting
    },

    # Walk-forward expanding-window folds (train_end is inclusive, test_start exclusive of train_end)
    'WALK_FORWARD_FOLDS': [
        {'train_start': '2016-04-18', 'train_end': '2020-12-31',
         'test_start':  '2021-01-01', 'test_end':  '2021-12-31'},
        {'train_start': '2016-04-18', 'train_end': '2021-12-31',
         'test_start':  '2022-01-01', 'test_end':  '2022-12-31'},
        {'train_start': '2016-04-18', 'train_end': '2022-12-31',
         'test_start':  '2023-01-01', 'test_end':  '2026-04-18'},
    ],

    # HMM regime model (Stage 2 strategy #13)
    'HMM_N_STATES':      3,
    'HMM_EXPOSURE':      {'bull': 1.0, 'choppy': 0.5, 'bear': 0.0},

    # Evaluation
    'BOOTSTRAP_N':       1000,
    'BOOTSTRAP_BLOCK':   20,             # block length for stationary bootstrap
    'NEWEY_WEST_LAG':    5,

    # Reproducibility
    'SEED':              SEED,
}

random.seed(SEED)
np.random.seed(SEED)

log.info('RUN_CONFIG loaded: %d top-level keys, seed=%d',
         len(RUN_CONFIG), RUN_CONFIG['SEED'])

In [ ]:
# ── Version pins (audit trail) ────────────────────────────────────────────
# Printed once so the exact environment is recoverable from a saved run.

import importlib

_LIBS = [
    'pandas', 'numpy', 'sklearn', 'matplotlib', 'yfinance',
    'pandas_datareader', 'hmmlearn', 'statsmodels', 'xgboost', 'scipy',
]

print(f'Python           {sys.version.split()[0]}')
for name in _LIBS:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'{name:<18} {ver}')
    except ImportError:
        print(f'{name:<18} NOT INSTALLED')

In [ ]:
# ── Global matplotlib style (set once, inherited by every figure) ─────────

mpl.rcParams.update({
    'figure.figsize':       (12, 5),
    'figure.dpi':           110,
    'savefig.dpi':          160,
    'savefig.bbox':         'tight',
    'font.family':          'DejaVu Sans',
    'font.size':            10,
    'axes.titlesize':       12,
    'axes.titleweight':     'semibold',
    'axes.labelsize':       10,
    'axes.grid':            True,
    'axes.spines.top':      False,
    'axes.spines.right':    False,
    'axes.prop_cycle':      mpl.cycler(color=[
        '#0B3D91',   # deep navy
        '#C0392B',   # brick
        '#117A65',   # forest
        '#B9770E',   # burnt orange
        '#6C3483',   # plum
        '#7F8C8D',   # slate
    ]),
    'grid.alpha':           0.25,
    'grid.linestyle':       '--',
    'lines.linewidth':      1.6,
    'legend.frameon':       False,
    'legend.fontsize':      9,
})

log.info('matplotlib style configured')

---

## Stage 1 — Data Layer

Three data streams feed the entire project:

1. **DJIA constituents** — reconstructed day-by-day from an April-2016 baseline set plus the six index reconstitutions in the sample window, so no stock receives weight on a date it wasn't actually in the index (no survivorship bias).
2. **Prices & volume** — `yfinance` auto-adjusted daily (splits + cash distributions folded into the close), forward-filled up to `FFILL_LIMIT_DAYS` trading days for short gaps. One post-2024 delisting (`WBA`, taken private by Sycamore Partners in 2025) is dropped from the download universe; this reduces the effective membership by one name during the 2018-06-26 → 2024-02-26 window in which WBA was an index member.
3. **Risk-free rate** — FRED `DTB3` (3-month T-bill secondary-market yield, annualised %), converted to a daily rate via `(r/100)/252` and forward-filled to every business day.

Each stream is wrapped in a pure function driven by `RUN_CONFIG`, so Stage 1 can be re-run deterministically without touching downstream code.

In [ ]:
# ── DJIA 30 constituent reconstruction ──────────────────────────────────
# Baseline as of April 2016; explicit changes capture the six known index
# reconstitutions in the ten-year sample window. Effective dates and
# ticker replacements cross-checked against S&P DJI press releases.

BASELINE_APR2016: set[str] = {
    'AAPL', 'AXP', 'BA',  'CAT',  'CSCO', 'CVX',  'DD',
    'DIS',  'GE',  'GS',  'HD',   'IBM',  'INTC', 'JNJ',
    'JPM',  'KO',  'MCD', 'MMM',  'MRK',  'MSFT', 'NKE',
    'PFE',  'PG',  'RTX', 'TRV',  'UNH',  'V',    'VZ',
    'WMT',  'XOM',
}

# (effective_date, added_tickers, removed_tickers)
CHANGES: list[tuple[str, list[str], list[str]]] = [
    ('2018-06-26', ['WBA'],              ['GE']),
    ('2019-04-02', ['DOW'],              ['DD']),
    ('2020-04-06', ['RTX'],              ['UTX']),     # post-merger relabel (UTX→RTX)
    ('2020-08-31', ['AMGN','CRM','HON'], ['XOM','PFE','RTX']),
    ('2024-02-26', ['AMZN','SHW'],       ['WBA','INTC']),
    ('2024-11-01', ['NVDA'],             ['DOW']),
]

# WBA was taken private (Sycamore Partners, 2025) and no longer has a
# continuous yfinance history. Dropping it from the download universe
# costs one name on 2018-06-26 → 2024-02-26 but keeps the data pipeline
# deterministic. Documented in the ICM data-quality section.
EXCLUDE_FROM_DOWNLOAD: set[str] = {'WBA'}


def build_constituent_map(
    baseline: set[str],
    changes: list[tuple[str, list[str], list[str]]],
    start: str,
    end: str,
    exclude_from_download: set[str] = EXCLUDE_FROM_DOWNLOAD,
) -> tuple[pd.DataFrame, list[str]]:
    """Reconstruct day-by-day DJIA membership over a business-day index.

    Args:
        baseline: tickers in the index on the first date of the window.
        changes: ordered reconstitutions as (effective_date, added, removed).
        start, end: window endpoints (inclusive) in YYYY-MM-DD form.
        exclude_from_download: tickers to strip from the column set because
            price data is unavailable. They are still honored historically
            — the effective universe is just one name smaller during
            periods when they were members.

    Returns:
        (constituent_map, all_tickers) where constituent_map is a
        (business_day × ticker) boolean DataFrame and all_tickers is the
        sorted column list with excluded tickers removed.
    """
    days = pd.bdate_range(start, end)
    changes_sorted = sorted(changes, key=lambda x: pd.Timestamp(x[0]))

    current = set(baseline)
    change_idx = 0
    rows: list[tuple[pd.Timestamp, str]] = []

    for day in days:
        while change_idx < len(changes_sorted):
            eff_date = pd.Timestamp(changes_sorted[change_idx][0])
            if eff_date <= day:
                _, added, removed = changes_sorted[change_idx]
                current.update(added)
                current.difference_update(removed)
                change_idx += 1
            else:
                break
        for ticker in current:
            rows.append((day, ticker))

    df = pd.DataFrame(rows, columns=['date', 'ticker'])
    all_tickers = sorted(set(df['ticker']) - exclude_from_download)

    cmap = pd.DataFrame(False, index=days, columns=all_tickers)
    for date, group in df.groupby('date'):
        in_index = [t for t in group['ticker'] if t in all_tickers]
        cmap.loc[date, in_index] = True
    return cmap, all_tickers


cfg = RUN_CONFIG   # short alias used throughout
constituent_map, ALL_TICKERS = build_constituent_map(
    BASELINE_APR2016, CHANGES, cfg['START_DATE'], cfg['END_DATE']
)
log.info('Constituent map: %d business days × %d tickers (WBA excluded from download)',
         *constituent_map.shape)


In [ ]:
# ── yfinance and FRED wrappers ────────────────────────────────────────────

import yfinance as yf
import pandas_datareader.data as web


def download_prices_volume(
    tickers: list[str],
    start: str,
    end: str,
    ffill_limit: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Pull adjusted close + volume from yfinance for `tickers`.

    `auto_adjust=True` folds splits and cash distributions into the close,
    so the returned price panel is directly usable for return construction.
    Short gaps (≤ `ffill_limit` trading days) are carried forward to absorb
    the occasional missing tick; longer gaps are preserved as NaN and
    surface in the Stage 1 validation report.
    """
    raw = yf.download(tickers, start=start, end=end,
                      auto_adjust=True, progress=False)
    prices = raw['Close'].sort_index().ffill(limit=ffill_limit)
    volume = raw['Volume'].sort_index().ffill(limit=ffill_limit)
    return prices, volume


def load_risk_free(start: str, end: str) -> pd.Series:
    """Fetch FRED DTB3 and convert to a daily risk-free rate series.

    FRED reports DTB3 as an annualised percentage (secondary-market yield
    on the 3-month T-bill). We approximate the daily rate by `(r/100)/252`;
    the error versus true geometric compounding is < 1 bp at DTB3 levels
    observed in sample.
    """
    raw = web.DataReader('DTB3', 'fred', start, end).squeeze()
    daily = (raw / 100.0) / 252.0
    biz_days = pd.date_range(start, end, freq='B')
    return daily.reindex(biz_days).ffill()


In [ ]:
# ── Execute the data pipeline ─────────────────────────────────────────────
# prices, volume, rf_daily are module-level globals consumed by every
# downstream stage. Rerunning this cell is the single point of refresh.

log.info('Downloading %d tickers from yfinance (%s → %s)...',
         len(ALL_TICKERS), cfg['START_DATE'], cfg['END_DATE'])

prices, volume = download_prices_volume(
    ALL_TICKERS,
    cfg['START_DATE'],
    cfg['END_DATE'],
    cfg['FFILL_LIMIT_DAYS'],
)

rf_daily = load_risk_free(cfg['START_DATE'], cfg['END_DATE'])

# Align the constituent map to the actual trading-day index from yfinance
# (business days minus US holidays). Reindex preserves only dates for
# which we have price data; gaps between b-days collapse cleanly.
constituent_map = (
    constituent_map.reindex(prices.index).ffill().fillna(False).astype(bool)
)

log.info('Prices  : %s', prices.shape)
log.info('Volume  : %s', volume.shape)
log.info('RF rate : %s  avg %.2f%% annualised',
         rf_daily.shape, rf_daily.mean() * 252 * 100)
log.info('Const.  : %s  avg %.1f stocks/day',
         constituent_map.shape, constituent_map.sum(axis=1).mean())


In [ ]:
# ── Data validation ───────────────────────────────────────────────────────

def data_validation_report(
    prices: pd.DataFrame,
    volume: pd.DataFrame,
    constituent_map: pd.DataFrame,
) -> pd.DataFrame:
    """Per-ticker summary of coverage, gaps, and constituent tenure.

    Flags tickers whose largest consecutive missing-price run exceeds
    the ffill limit — candidates for exclusion and worth a note in the
    ICM data-quality section.
    """
    rows = []
    for t in prices.columns:
        px = prices[t]
        is_member = constituent_map.get(t, pd.Series(False, index=prices.index))
        member_days = int(is_member.sum())
        na_mask = px[is_member].isna()
        na_count = int(na_mask.sum())
        if na_mask.any():
            groups = (na_mask != na_mask.shift()).cumsum()
            max_gap = int(na_mask.groupby(groups).sum().max())
        else:
            max_gap = 0
        rows.append({
            'ticker':       t,
            'first_member': is_member.idxmax() if member_days else pd.NaT,
            'last_member':  is_member[::-1].idxmax() if member_days else pd.NaT,
            'member_days':  member_days,
            'na_in_tenure': na_count,
            'na_pct':       (na_count / member_days) if member_days else np.nan,
            'max_gap_days': max_gap,
        })
    out = pd.DataFrame(rows).set_index('ticker')
    return out.sort_values('max_gap_days', ascending=False)


report = data_validation_report(prices, volume, constituent_map)
print(f"Price panel  : {prices.shape[0]:>5} rows × {prices.shape[1]:>2} cols")
print(f"Date range   : {prices.index.min().date()} → {prices.index.max().date()}")
print(f"Avg #stocks  : {constituent_map.sum(axis=1).mean():.1f} per day")
print(f"Min #stocks  : {constituent_map.sum(axis=1).min()} on "
      f"{constituent_map.sum(axis=1).idxmin().date()}")
print(f"Max in-tenure gap : {report['max_gap_days'].max()} trading days "
      f"(ticker={report['max_gap_days'].idxmax()})")

dropped = report[report['max_gap_days'] > cfg['FFILL_LIMIT_DAYS']]
if len(dropped):
    print(f"\nTickers with in-tenure gaps > {cfg['FFILL_LIMIT_DAYS']}d (review):")
    print(dropped[['member_days', 'na_in_tenure', 'max_gap_days']])
else:
    print(f"\nAll in-tenure gaps ≤ {cfg['FFILL_LIMIT_DAYS']} trading days ✓")

report.head(15)


In [ ]:
# ── Universe composition over time ────────────────────────────────────────
# Left: rolling count of index members by day (reconstitutions show as
# step changes). Right: Gantt-style membership tenure per ticker — the
# six reconstitution events become easy to eyeball.

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 1.3]}
)

n_members = constituent_map.sum(axis=1)
ax1.plot(n_members.index, n_members.values, color='#0B3D91', linewidth=1.4)
ax1.axhline(30, color='gray', linestyle=':', linewidth=0.9, label='Target (30)')
ax1.fill_between(n_members.index, 28, 30, alpha=0.05, color='#0B3D91')
ax1.set_title('DJIA membership count over sample window')
ax1.set_ylabel('Number of constituents')
ax1.set_ylim(26, 31)
ax1.legend(loc='lower left')

sorted_tickers = (
    constituent_map.sum(axis=0).sort_values(ascending=True).index.tolist()
)
for i, t in enumerate(sorted_tickers):
    mask = constituent_map[t]
    if not mask.any():
        continue
    grp = (mask != mask.shift()).cumsum()
    for _, run in mask.groupby(grp):
        if run.iloc[0]:
            ax2.hlines(i, run.index.min(), run.index.max(),
                       colors='#0B3D91', linewidth=3.0, alpha=0.85)
ax2.set_yticks(range(len(sorted_tickers)))
ax2.set_yticklabels(sorted_tickers, fontsize=7)
ax2.set_title('Membership tenure by ticker')
ax2.set_xlim(pd.Timestamp(cfg['START_DATE']), pd.Timestamp(cfg['END_DATE']))
ax2.grid(axis='x', alpha=0.25, linestyle='--')
ax2.set_axisbelow(True)

plt.tight_layout()
plt.show()


---

## Stage 2 — Strategy Zoo (15 momentum variants)

Every strategy is a pure function with the signature
`strategy_name(prices, volume, constituent_map, cfg) -> weights_df`.
Weights are daily with monthly rebalancing (month-end snapshot
forward-filled to every trading day). The long leg is the top
`cfg['LONG_PCT']` quintile, short leg the bottom `cfg['SHORT_PCT']`
quintile, equal-weighted inside each; deviations (TSMOM, vol-scaled)
are flagged per strategy.

**Taxonomy.**

1. **Classical** (1-6) — canonical cross-sectional and time-series
   momentum from the academic literature: Jegadeesh & Titman (1993),
   Moskowitz/Ooi/Pedersen (2012), Antonacci (2012).
2. **Advanced** (7-12) — signal-quality improvements: risk-adjustment,
   residualisation (Blitz/Huij/Martens 2011), sector-neutrality,
   reversal overlays, acceleration, volume confirmation.
3. **Regime-aware / ML** (13-15) — HMM-gated exposure
   (Daniel & Moskowitz 2016 spirit), XGBoost stacking, rank ensemble.

**Invariants enforced by the shared helpers.**

- Non-constituents receive `NaN` signal and therefore zero weight on dates they weren't in the index.
- Dollar-neutral portfolios: long weights sum to +1, short to −1; total gross exposure ≤ 2.
- No look-ahead: `backtest()` shifts weights by one trading day before multiplying by realised returns, so signal at date *t* produces a trade at *t+1*.
- Transaction costs are charged on `|Δw|` at `cfg['COST_BPS']` (10 bps base case).

Stage 2 is delivered across multiple commits (≈ 3 strategies each per PROMPT.md §6 step 4). This commit lays down the shared infrastructure and the three classical cross-sectional-momentum variants.

In [ ]:
# ── Shared helpers consumed by every strategy ─────────────────────────────

def signal_to_weights(
    signal: pd.DataFrame,
    constituent_map: pd.DataFrame,
    long_pct: float,
    short_pct: float,
) -> pd.DataFrame:
    """Rank a cross-sectional signal into market-neutral long/short weights.

    Higher signal -> longer position. Equal-weighted inside each quintile.
    Non-constituents are masked to NaN before ranking so dropped names do
    not consume quintile slots.

    Args:
        signal:          (date x ticker) DataFrame. Higher = more long.
        constituent_map: (date x ticker) boolean membership mask.
        long_pct:        top fraction to go long (e.g., 0.20 = top quintile).
        short_pct:       bottom fraction to short.

    Returns:
        Weights DataFrame summing to ~0 cross-sectionally with |weights|
        summing to ~2 when both legs are populated.
    """
    mask = constituent_map.reindex_like(signal).fillna(False)
    sig = signal.where(mask)
    ranks = sig.rank(axis=1, pct=True)
    long_mask  = (ranks >= 1 - long_pct).astype(float)
    short_mask = (ranks <= short_pct).astype(float)
    n_long  = long_mask.sum(axis=1).replace(0, np.nan)
    n_short = short_mask.sum(axis=1).replace(0, np.nan)
    long_w  =  long_mask.div(n_long,  axis=0)
    short_w = -short_mask.div(n_short, axis=0)
    return (long_w + short_w).fillna(0)


def monthly_rebalance(weights_daily: pd.DataFrame, freq: str) -> pd.DataFrame:
    """Sample weights at `freq` (e.g. 'ME') and hold constant until next rebalance."""
    return (
        weights_daily.resample(freq).last()
        .reindex(weights_daily.index)
        .ffill()
        .fillna(0)
    )


def backtest(
    weights: pd.DataFrame,
    prices: pd.DataFrame,
    cost_bps: float,
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """Compute gross/net daily P&L under no-look-ahead and turnover cost.

    Signal at day t produces trades at t+1: weights are shifted by one day
    before the return dot product. Cost at day t = |Δweights[t]| × bps/10000.
    Charging on raw `|Δw|` (rather than dollar notional) is the standard
    L/S equity-factor convention and matches the baseline notebook.
    """
    asset_rets = prices.pct_change()
    traded = weights.shift(1)
    gross = (traded * asset_rets).sum(axis=1, min_count=1).fillna(0)
    turnover = weights.diff().abs().sum(axis=1).fillna(0)
    cost = turnover * (cost_bps / 10000.0)
    net = gross - cost.reindex(gross.index).fillna(0)
    return gross, net, turnover


def sanity_check_weights(weights: pd.DataFrame, name: str) -> None:
    """Log warnings on market-neutrality or leverage violations. Non-raising."""
    row_sum = weights.sum(axis=1).abs()
    gross = weights.abs().sum(axis=1)
    if row_sum.max() > 0.01:
        log.warning('%s: row sums not dollar-neutral (max |sum(w)|=%.3f)',
                    name, row_sum.max())
    if gross.max() > 2.01:
        log.warning('%s: leverage exceeded (max sum|w|=%.3f)', name, gross.max())
    if weights.isna().any().any():
        log.warning('%s: weights contain NaN', name)


def quick_summary(net: pd.Series, turnover: pd.Series, name: str) -> dict:
    """One-line headline metrics for Stage 2 spot-checks."""
    if net.std() == 0 or len(net) == 0:
        return {'name': name, 'ann_ret': 0.0, 'ann_vol': 0.0,
                'sharpe': np.nan, 'turnover_yr': 0.0}
    excess = net - rf_daily.reindex(net.index).fillna(0)
    ann_ret = (1 + net).prod() ** (252 / len(net)) - 1
    ann_vol = net.std() * np.sqrt(252)
    sharpe  = excess.mean() / net.std() * np.sqrt(252)
    return {
        'name': name,
        'ann_ret': ann_ret,
        'ann_vol': ann_vol,
        'sharpe':  sharpe,
        'turnover_yr': turnover.mean() * 252,
    }


In [ ]:
# ── Strategies 1-3 · Classical cross-sectional momentum ───────────────────
# Canonical Jegadeesh & Titman (1993) formulation with a skip-one-month lag
# (21 trading days). The three variants use 12-, 6-, and 3-month look-back
# windows to test whether intermediate-horizon continuation dominates shorter
# reversal or longer reversion effects in the DJIA universe.


def strat_cs_mom_12_1(prices, volume, constituent_map, cfg):
    """#1  Classical 12-1 cross-sectional momentum (Jegadeesh & Titman 1993)."""
    lb = cfg['LOOKBACKS']
    signal = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_cs_mom_6_1(prices, volume, constituent_map, cfg):
    """#2  6-1 cross-sectional momentum — shorter look-back, higher turnover."""
    lb = cfg['LOOKBACKS']
    signal = prices.shift(lb['skip']) / prices.shift(lb['mom_mid']) - 1
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_cs_mom_3_1(prices, volume, constituent_map, cfg):
    """#3  3-1 cross-sectional momentum — shortest classical horizon."""
    lb = cfg['LOOKBACKS']
    signal = prices.shift(lb['skip']) / prices.shift(lb['mom_short']) - 1
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


In [ ]:
# ── Spot-check classical variants (full-sample, pre-walk-forward) ─────────
# Stage 3 will enforce expanding-window out-of-sample evaluation; this cell
# is a sanity run so any construction bug shows up before we've committed
# 12 more strategies on top.

_strategies_so_far = {
    'cs_mom_12_1': strat_cs_mom_12_1,
    'cs_mom_6_1':  strat_cs_mom_6_1,
    'cs_mom_3_1':  strat_cs_mom_3_1,
}

STRATEGIES: dict = globals().get('STRATEGIES', {})
RESULTS: dict = globals().get('RESULTS', {})

_rows = []
for _name, _fn in _strategies_so_far.items():
    _w = _fn(prices, volume, constituent_map, cfg)
    sanity_check_weights(_w, _name)
    _gross, _net, _to = backtest(_w, prices, cfg['COST_BPS'])
    STRATEGIES[_name] = _fn
    RESULTS[_name] = {'weights': _w, 'gross': _gross, 'net': _net, 'turnover': _to}
    _rows.append(quick_summary(_net, _to, _name))

_spot = pd.DataFrame(_rows).set_index('name')
_spot[['ann_ret','ann_vol','sharpe','turnover_yr']].style.format({
    'ann_ret':      '{:.2%}',
    'ann_vol':      '{:.2%}',
    'sharpe':       '{:.2f}',
    'turnover_yr':  '{:.1f}x',
})


---

## Stage 3 — Evaluation Framework & Leaderboard

_Single `evaluate()` producing risk-adjusted metrics, FF3 regression, bootstrap Sharpe CI, Newey-West t-stats. Master leaderboard sorted by walk-forward OOS Sharpe. Implemented in the next stage commit._

---

## Stage 4 — Diagnostic Visuals (Top 5)

_Cumulative curves, rolling Sharpe, underwater, return distribution, monthly heatmap, FF3 exposure, turnover. Top 5 by OOS Sharpe only. Implemented in the next stage commit._

---

## Stage 5 — Iterative Improvement Loop

_Up to five rounds of diagnose → propose → implement → compare → decide, starting from the #1 ranked strategy. Early-stop when OOS Sharpe gain < 0.05. Implemented in the next stage commit._

---

## Stage 6 — Robustness Tests (Winner)

_Cost/frequency/lookback sensitivities, subperiod stability, Monte Carlo bootstrap, stress-period P&L, capacity estimate. Implemented in the next stage commit._

---

## Stage 7 — Investment Committee Memorandum

_Executive summary, strategy rationale, P&L conditions, statistical properties, 10-year performance review, risk factors, recommendation, Appendix A (source attribution incl. verbatim PROMPT.md and `llm_interactions.log`), Appendix B (iteration log). Written in the next stage commit._